# Margin-Template Rule Ensembles: Synthetic Benchmark and Interpretation

Evaluates Student-t Margin Template Ensembles (SMTE) and Gaussian Margin Template Ensembles (GMTE) on controlled synthetic datasets with known ground-truth features, interactions, and rules.

- Feature recovery evaluated by gradient saliency.
- Interaction recovery evaluated by gradient-gradient covariance.
- Rule examples extracted from top head-weighted templates.
- Interpretability score combines feature recovery, displayed rule sparsity, interaction detection, and sign/direction recovery.

Baselines: EBM, XGBoost, CatBoost, RuleFit.

## 0. Dependencies

In [1]:
!pip install -q numpy pandas scikit-learn matplotlib seaborn torch xgboost catboost interpret imodels

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 42.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.5/274.5 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 14.8 MB/s eta 0:00:00


## 1. Imports and configuration

In [2]:
import os
import re
import json
import time
import math
import random
import warnings
import platform
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    f1_score,
)
from sklearn.multiclass import OneVsRestClassifier

try:
    from xgboost import XGBRegressor, XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    from catboost import CatBoostRegressor, CatBoostClassifier
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False

try:
    from interpret.glassbox import ExplainableBoostingRegressor, ExplainableBoostingClassifier
    HAS_EBM = True
except Exception:
    HAS_EBM = False

try:
    from imodels import RuleFitRegressor, RuleFitClassifier
    HAS_RULEFIT = True
except Exception:
    HAS_RULEFIT = False

warnings.filterwarnings("ignore")

# Reproducibility.
SEED = 42

SEEDS = [1, 12, 123, 1234, 12345]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUTPUT_DIR = Path("outputs_synthetic_paper")
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"

for d in [OUTPUT_DIR, TABLE_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

N_MAIN = 2000
D_MAIN = 15

N_RULES = 32
EPOCHS = 150
PATIENCE = 20
BATCH_SIZE = 256
LR = 3e-3
WEIGHT_DECAY = 1e-4
L1_MASK = 1e-4
RULE_BETA = 6.0

TRAIN_TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAIN = 0.20

# Interpretation settings.
GRADIENT_N_SAMPLES = 300
TOP_RULES_DISPLAY = 5
TOP_FEATURES_REPORT = 8
RULE_DISPLAY_ABSOLUTE_FLOOR = 0.05
RULE_DISPLAY_RELATIVE_FRACTION = 0.20

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:.4f}".format)

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def now() -> float:
    return time.perf_counter()

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mean_std_str(x: pd.Series, digits: int = 3) -> str:
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) == 0:
        return "nan ± nan"
    if len(x) == 1:
        return f"{x.mean():.{digits}f} ± nan"
    return f"{x.mean():.{digits}f} ± {x.std(ddof=1):.{digits}f}"

def set_paper_style() -> None:
    sns.set_theme(style="whitegrid", context="paper")
    plt.rcParams.update(
        {
            "figure.dpi": 300,
            "savefig.dpi": 600,
            "font.family": "serif",
            "font.size": 10,
            "axes.titlesize": 11,
            "axes.labelsize": 10,
            "legend.fontsize": 9,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )

def savefig(name: str) -> None:
    path = FIG_DIR / f"{name}.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        plt.tight_layout()
    except Exception:
        pass
    plt.savefig(path, bbox_inches="tight")
    plt.close()

set_seed(SEED)
set_paper_style()

MODEL_COLORS = {
    "SMTE": "#3B82F6",
    "GMTE": "#06B6D4",
    "EBM": "#8B5CF6",
    "XGBoost": "#6B7280",
    "CatBoost": "#F59E0B",
    "RuleFit": "#10B981",
}

print(f"Device        : {DEVICE}")
print(f"Seeds         : {SEEDS}")
print(f"Output dir    : {OUTPUT_DIR}")
print(f"XGBoost       : {HAS_XGBOOST}")
print(f"CatBoost      : {HAS_CATBOOST}")
print(f"EBM           : {HAS_EBM}")
print(f"RuleFit       : {HAS_RULEFIT}")

Device        : cuda
Seeds         : [1, 12, 123, 1234, 12345]
Output dir    : outputs_synthetic_paper
XGBoost       : True
CatBoost      : True
EBM           : True
RuleFit       : True


## 2. Environment table

In [3]:
env_table = pd.DataFrame(
    [
        {
            "python": platform.python_version(),
            "platform": platform.platform(),
            "processor": platform.processor(),
            "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "device": str(DEVICE),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "sklearn": __import__("sklearn").__version__,
            "xgboost_available": HAS_XGBOOST,
            "catboost_available": HAS_CATBOOST,
            "ebm_available": HAS_EBM,
            "rulefit_available": HAS_RULEFIT,
            "seeds": str(SEEDS),
            "n_main": N_MAIN,
            "d_main": D_MAIN,
            "n_rules": N_RULES,
            "epochs": EPOCHS,
            "patience": PATIENCE,
            "batch_size": BATCH_SIZE,
            "learning_rate": LR,
            "weight_decay": WEIGHT_DECAY,
            "l1_mask": L1_MASK,
            "rule_beta": RULE_BETA,
            "gradient_n_samples": GRADIENT_N_SAMPLES,
            "rule_display_absolute_floor": RULE_DISPLAY_ABSOLUTE_FLOOR,
            "rule_display_relative_fraction": RULE_DISPLAY_RELATIVE_FRACTION,
        }
    ]
)

env_table.to_csv(TABLE_DIR / "table_environment.csv", index=False)
display(env_table)

,python,platform,processor,torch,cuda_available,device,numpy,pandas,sklearn,xgboost_available,catboost_available,ebm_available,rulefit_available,seeds,n_main,d_main,n_rules,epochs,patience,batch_size,learning_rate,weight_decay,l1_mask,rule_beta,gradient_n_samples,rule_display_absolute_floor,rule_display_relative_fraction
0,3.12.12,Linux-6.6.122+-x86_64-with-glibc2.35,x86_64,2.10.0+cu128,True,cuda,2.0.2,2.3.3,1.6.1,True,True,True,True,"[1, 12, 123, 1234, 12345]",2000,15,32,150,20,256,0.0030,0.0001,0.0001,6.0000,300,0.0500,0.2000


## 3. Synthetic datasets

In [4]:
SYNTHETIC_DATASETS: Dict[str, dict] = {}

def make_hierarchical(n: int = N_MAIN, d: int = D_MAIN, noise: float = 0.3, seed: int = SEED):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, (n, d)).astype(np.float32)

    y = np.zeros(n, dtype=np.float32)
    y += 3.0 * ((X[:, 0] > 0.5) & (X[:, 1] < 0)).astype(np.float32)
    y += 2.0 * (X[:, 2] > 1).astype(np.float32) * X[:, 5]
    y += 1.5 * X[:, 8]
    y += 0.8 * X[:, 0] * X[:, 1]
    y += noise * rng.standard_normal(n).astype(np.float32)

    return pd.DataFrame(X, columns=[f"x{i}" for i in range(d)]), pd.Series(y)

SYNTHETIC_DATASETS["Hierarchical"] = dict(
    loader=make_hierarchical,
    task="regression",
    true_rule="IF x0>0.5 AND x1<0 THEN +3; IF x2>1 THEN +2*x5; plus 1.5*x8 and 0.8*x0*x1",
    active_feats={0: None, 1: None, 2: None, 5: None, 8: +1.5},
    n_active=5,
    known_positive_features={8},
    known_negative_features=set(),
)

def make_friedman1(n: int = N_MAIN, d: int = D_MAIN, noise: float = 0.3, seed: int = SEED):
    rng = np.random.default_rng(seed)
    X = rng.uniform(0, 1, (n, d)).astype(np.float32)

    y = (
        10.0 * np.sin(np.pi * X[:, 0] * X[:, 1])
        + 20.0 * (X[:, 2] - 0.5) ** 2
        + 10.0 * X[:, 3]
        + 5.0 * X[:, 4]
        + noise * rng.standard_normal(n).astype(np.float32)
    ).astype(np.float32)

    return pd.DataFrame(X, columns=[f"x{i}" for i in range(d)]), pd.Series(y)

SYNTHETIC_DATASETS["Friedman1"] = dict(
    loader=make_friedman1,
    task="regression",
    true_rule="10*sin(pi*x0*x1) + 20*(x2-0.5)^2 + 10*x3 + 5*x4",
    active_feats={0: None, 1: None, 2: None, 3: +10.0, 4: +5.0},
    n_active=5,
    known_positive_features={3, 4},
    known_negative_features=set(),
)

def make_xor_chain(n: int = N_MAIN, d: int = D_MAIN, seed: int = SEED):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, (n, d)).astype(np.float32)

    y = np.zeros(n, dtype=np.int64)
    y[((X[:, 0] > 0) ^ (X[:, 1] > 0))] = 1
    y[(X[:, 2] > 0.5) & (X[:, 3] < -0.5)] = 2

    return pd.DataFrame(X, columns=[f"x{i}" for i in range(d)]), pd.Series(y)

SYNTHETIC_DATASETS["XorChain"] = dict(
    loader=make_xor_chain,
    task="classification",
    true_rule="class 1: x0 XOR x1; class 2: x2>0.5 AND x3<-0.5",
    active_feats={0: None, 1: None, 2: None, 3: None},
    n_active=4,
    known_positive_features=set(),
    known_negative_features=set(),
)

def make_threshold_cls(n: int = N_MAIN, d: int = D_MAIN, seed: int = SEED):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, (n, d)).astype(np.float32)

    score = (
        2.0 * (X[:, 0] > 0.3).astype(np.float32)
        - 1.5 * (X[:, 4] > 0).astype(np.float32)
        + X[:, 7]
    )
    y = (score > score.mean()).astype(np.int64)

    return pd.DataFrame(X, columns=[f"x{i}" for i in range(d)]), pd.Series(y)

SYNTHETIC_DATASETS["ThresholdCls"] = dict(
    loader=make_threshold_cls,
    task="classification",
    true_rule="score = 2*1{x0>0.3} - 1.5*1{x4>0} + x7",
    active_feats={0: None, 4: None, 7: +1.0},
    n_active=3,
    known_positive_features={7},
    known_negative_features=set(),
)

def make_multithresh(n: int = N_MAIN, d: int = D_MAIN, seed: int = SEED):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, (n, d)).astype(np.float32)

    y = np.zeros(n, dtype=np.int64)
    y[(X[:, 0] > 0) & (X[:, 1] > 0)] = 1
    y[(X[:, 0] < 0) & (X[:, 1] > 0) & (X[:, 2] > 0.5)] = 2
    y[(X[:, 6] > 1.0)] = 3

    return pd.DataFrame(X, columns=[f"x{i}" for i in range(d)]), pd.Series(y)

SYNTHETIC_DATASETS["MultiThresh"] = dict(
    loader=make_multithresh,
    task="classification",
    true_rule="class 1: x0>0 AND x1>0; class 2: x0<0 AND x1>0 AND x2>0.5; class 3: x6>1",
    active_feats={0: None, 1: None, 2: None, 6: None},
    n_active=4,
    known_positive_features=set(),
    known_negative_features=set(),
)

dataset_rows = []

for name, meta in SYNTHETIC_DATASETS.items():
    X, y = meta["loader"](seed=SEED)
    dataset_rows.append(
        {
            "dataset": name,
            "task": meta["task"],
            "n": X.shape[0],
            "d": X.shape[1],
            "n_active": meta["n_active"],
            "active_features": str([f"x{i}" for i in meta["active_feats"].keys()]),
            "true_rule": meta["true_rule"],
        }
    )
    print(f"{name:18s} | task={meta['task']:14s} | X={X.shape} | active={meta['n_active']}")

dataset_table = pd.DataFrame(dataset_rows)
dataset_table.to_csv(TABLE_DIR / "table_s1_dataset_specification.csv", index=False)
display(dataset_table)

Hierarchical       | task=regression     | X=(2000, 15) | active=5
Friedman1          | task=regression     | X=(2000, 15) | active=5
XorChain           | task=classification | X=(2000, 15) | active=4
ThresholdCls       | task=classification | X=(2000, 15) | active=3
MultiThresh        | task=classification | X=(2000, 15) | active=4


,dataset,task,n,d,n_active,active_features,true_rule
0,Hierarchical,regression,2000,15,5,"['x0', 'x1', 'x2', 'x5', 'x8']",IF x0>0.5 AND x1<0 THEN +3; IF x2>1 THEN +2*x5...
1,Friedman1,regression,2000,15,5,"['x0', 'x1', 'x2', 'x3', 'x4']",10*sin(pi*x0*x1) + 20*(x2-0.5)^2 + 10*x3 + 5*x4
2,XorChain,classification,2000,15,4,"['x0', 'x1', 'x2', 'x3']",class 1: x0 XOR x1; class 2: x2>0.5 AND x3<-0.5
3,ThresholdCls,classification,2000,15,3,"['x0', 'x4', 'x7']",score = 2*1{x0>0.3} - 1.5*1{x4>0} + x7
4,MultiThresh,classification,2000,15,4,"['x0', 'x1', 'x2', 'x6']",class 1: x0>0 AND x1>0; class 2: x0<0 AND x1>0...


## 4. Prepared-data container

In [5]:
@dataclass
class PreparedData:
    dataset: str
    seed: int
    task: str
    feature_names: List[str]
    n_classes: int

    X_train: np.ndarray
    X_val: np.ndarray
    X_test: np.ndarray

    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray

    y_train_scaled: np.ndarray
    y_val_scaled: np.ndarray

    y_scaler: Optional[StandardScaler]
    x_scaler: StandardScaler

def prepare_dataset(ds_name: str, seed: int = SEED) -> PreparedData:
    meta = SYNTHETIC_DATASETS[ds_name]
    X, y = meta["loader"](seed=seed)
    feature_names = list(X.columns)

    strat = y if meta["task"] == "classification" else None

    X_trva, X_te, y_trva, y_te = train_test_split(
        X,
        y,
        test_size=TRAIN_TEST_SIZE,
        random_state=seed,
        stratify=strat,
    )

    strat2 = y_trva if meta["task"] == "classification" else None

    X_tr, X_va, y_tr, y_va = train_test_split(
        X_trva,
        y_trva,
        test_size=VAL_SIZE_WITHIN_TRAIN,
        random_state=seed,
        stratify=strat2,
    )

    x_scaler = StandardScaler()
    X_train = x_scaler.fit_transform(X_tr).astype(np.float32)
    X_val = x_scaler.transform(X_va).astype(np.float32)
    X_test = x_scaler.transform(X_te).astype(np.float32)

    if meta["task"] == "regression":
        y_scaler = StandardScaler()
        y_train_scaled = y_scaler.fit_transform(y_tr.to_numpy().reshape(-1, 1)).ravel().astype(np.float32)
        y_val_scaled = y_scaler.transform(y_va.to_numpy().reshape(-1, 1)).ravel().astype(np.float32)

        y_train = y_tr.to_numpy().astype(np.float32)
        y_val = y_va.to_numpy().astype(np.float32)
        y_test = y_te.to_numpy().astype(np.float32)

        n_classes = 1
    else:
        y_scaler = None

        y_train = y_tr.to_numpy().astype(np.int64)
        y_val = y_va.to_numpy().astype(np.int64)
        y_test = y_te.to_numpy().astype(np.int64)

        y_train_scaled = y_train.copy()
        y_val_scaled = y_val.copy()

        n_classes = int(len(np.unique(y_train)))

    return PreparedData(
        dataset=ds_name,
        seed=seed,
        task=meta["task"],
        feature_names=feature_names,
        n_classes=n_classes,
        X_train=X_train,
        X_val=X_val,
        X_test=X_test,
        y_train=y_train,
        y_val=y_val,
        y_test=y_test,
        y_train_scaled=y_train_scaled,
        y_val_scaled=y_val_scaled,
        y_scaler=y_scaler,
        x_scaler=x_scaler,
    )

## 5. Margin-rule layer and proposed models

In [6]:
@torch.no_grad()
def initialize_thresholds(
    th_tensor: torch.Tensor,
    X: np.ndarray,
    q_lo: float = 0.10,
    q_hi: float = 0.90,
    max_q: int = 9,
) -> None:
    R, D = th_tensor.shape
    X = np.asarray(X, dtype=np.float32)

    for j in range(D):
        qs = np.linspace(q_lo, q_hi, min(R, max_q), dtype=np.float32)
        vals = np.quantile(X[:, j], qs).astype(np.float32)
        vals = np.resize(vals, R)
        th_tensor[:, j].copy_(torch.tensor(vals, dtype=torch.float32, device=th_tensor.device))

def log_margin(u: torch.Tensor, v: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    return torch.log(u + eps) - torch.log(v + eps)

class DualHeadMixin:
    def make_head(self, n_rules: int, n_classes: int):
        self.n_classes = n_classes
        return nn.Linear(n_rules, max(n_classes, 1))

    def supervised_loss(self, logits: torch.Tensor, y: torch.Tensor):
        if self.n_classes > 1:
            return F.cross_entropy(logits, y.long())
        return F.mse_loss(logits.view(-1), y.float())

class SignedRuleLayer(nn.Module):
    def __init__(self, input_dim: int, n_rules: int = N_RULES, beta: float = RULE_BETA):
        super().__init__()

        self.input_dim = input_dim
        self.n_rules = n_rules
        self.beta = beta

        self.th = nn.Parameter(torch.zeros(n_rules, input_dim))
        self.ineq = nn.Parameter(torch.randn(n_rules, input_dim) * 0.1)
        self.esign = nn.Parameter(torch.randn(n_rules, input_dim) * 0.1)
        self.mask_logit = nn.Parameter(torch.full((n_rules, input_dim), -2.0))
        self.log_kappa = nn.Parameter(torch.tensor(math.log(6.0)))
        self.t = nn.Parameter(torch.zeros(n_rules))

    @torch.no_grad()
    def init_from_data(self, X: np.ndarray):
        initialize_thresholds(self.th, X)

    def literals(self, x: torch.Tensor):
        kappa = torch.exp(self.log_kappa).clamp(0.5, 50.0)

        direction = torch.tanh(self.ineq)[None, :, :]
        c = torch.sigmoid(kappa * direction * (x[:, None, :] - self.th[None, :, :]))

        mask = torch.sigmoid(self.mask_logit)[None, :, :]
        sign = torch.tanh(self.esign)[None, :, :]

        signed_literal = mask * sign * (2.0 * c - 1.0)

        positive_evidence = mask * (
            0.5 * (1.0 + sign) * c
            + 0.5 * (1.0 - sign) * (1.0 - c)
        )
        negative_evidence = mask - positive_evidence

        return positive_evidence, negative_evidence, signed_literal, mask

    def forward(self, x: torch.Tensor):
        _, _, signed_literal, _ = self.literals(x)
        return torch.sigmoid(self.beta * (signed_literal.sum(dim=2) - self.t[None, :]))

class GaussianMarginTemplateEnsemble(DualHeadMixin, nn.Module):
    display_name = "GMTE"

    def __init__(self, input_dim: int, n_rules: int = N_RULES, n_classes: int = 1):
        super().__init__()

        self.rules = SignedRuleLayer(input_dim=input_dim, n_rules=n_rules)
        self.loc = nn.Parameter(torch.zeros(n_rules, input_dim))
        self.scale = nn.Parameter(torch.ones(n_rules, input_dim))
        self.head = self.make_head(n_rules, n_classes)

    def init_from_data(self, X: np.ndarray):
        self.rules.init_from_data(X)

    def forward(self, x: torch.Tensor):
        u, v, _, _ = self.rules.literals(x)
        m = log_margin(u, v)

        s2 = F.softplus(self.scale)[None, :, :] ** 2
        energy = -((m - self.loc[None, :, :]) ** 2 / (2.0 * s2 + 1e-6)).sum(dim=2)

        z = torch.sigmoid(self.rules.beta * (energy - self.rules.t[None, :]))
        return self.head(z)

    def loss_batch(self, x: torch.Tensor, y: torch.Tensor):
        return self.supervised_loss(self.forward(x), y)

class StudentTMarginTemplateEnsemble(DualHeadMixin, nn.Module):
    display_name = "SMTE"

    def __init__(self, input_dim: int, n_rules: int = N_RULES, n_classes: int = 1):
        super().__init__()

        self.rules = SignedRuleLayer(input_dim=input_dim, n_rules=n_rules)
        self.mu = nn.Parameter(torch.zeros(n_rules, input_dim))
        self.log_s = nn.Parameter(torch.zeros(n_rules, input_dim))

        self.log_nu = nn.Parameter(torch.ones(n_rules) * math.log(5.0))

        self.head = self.make_head(n_rules, n_classes)

    def init_from_data(self, X: np.ndarray):
        self.rules.init_from_data(X)

    def nu(self) -> torch.Tensor:
        return F.softplus(self.log_nu).clamp(1.0, 50.0)

    def forward(self, x: torch.Tensor):
        u, v, _, _ = self.rules.literals(x)
        m = log_margin(u, v)

        nu = self.nu()
        s2 = F.softplus(self.log_s) ** 2 + 1e-6

        energy_per_dim = -((nu[None, :, None] + 1.0) / 2.0) * torch.log1p(
            (m - self.mu[None, :, :]) ** 2 / (nu[None, :, None] * s2[None, :, :])
        )

        energy = energy_per_dim.sum(dim=2)
        z = torch.sigmoid(self.rules.beta * (energy - self.rules.t[None, :]))
        return self.head(z)

    def loss_batch(self, x: torch.Tensor, y: torch.Tensor):
        return self.supervised_loss(self.forward(x), y)

PROPOSED_MODELS = {
    "SMTE": StudentTMarginTemplateEnsemble,
    "GMTE": GaussianMarginTemplateEnsemble,
}

ALL_MODELS = ["SMTE", "GMTE", "EBM", "XGBoost", "CatBoost", "RuleFit"]

## 6. Training and prediction utilities

In [7]:
def build_margin_model(name: str, pp: PreparedData):
    model = PROPOSED_MODELS[name](
        input_dim=pp.X_train.shape[1],
        n_rules=N_RULES,
        n_classes=pp.n_classes,
    ).to(DEVICE)

    model.init_from_data(pp.X_train)
    return model

def train_margin_model(model, pp: PreparedData) -> Dict[str, List[float]]:
    Xtr = torch.tensor(pp.X_train, dtype=torch.float32, device=DEVICE)

    if pp.task == "regression":
        ytr = torch.tensor(pp.y_train_scaled, dtype=torch.float32, device=DEVICE)
        yva = torch.tensor(pp.y_val_scaled, dtype=torch.float32, device=DEVICE)
    else:
        ytr = torch.tensor(pp.y_train_scaled, dtype=torch.long, device=DEVICE)
        yva = torch.tensor(pp.y_val_scaled, dtype=torch.long, device=DEVICE)

    Xva = torch.tensor(pp.X_val, dtype=torch.float32, device=DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)

    best_val = float("inf")
    best_state = None
    wait = 0

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
    }

    for epoch in range(EPOCHS):
        model.train()

        perm = torch.randperm(len(Xtr), device=DEVICE)
        train_losses = []

        for i in range(0, len(Xtr), BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            xb = Xtr[idx]
            yb = ytr[idx]

            opt.zero_grad()

            loss = model.loss_batch(xb, yb)
            loss = loss + L1_MASK * torch.sigmoid(model.rules.mask_logit).sum()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            opt.step()
            train_losses.append(float(loss.detach().cpu()))

        sched.step()

        model.eval()
        with torch.no_grad():
            val_loss = float(model.loss_batch(Xva, yva).detach().cpu())

        history["epoch"].append(epoch)
        history["train_loss"].append(float(np.mean(train_losses)))
        history["val_loss"].append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return history

@torch.no_grad()
def predict_margin_model(model, pp: PreparedData) -> np.ndarray:
    model.eval()
    X = torch.tensor(pp.X_test, dtype=torch.float32, device=DEVICE)
    out = model(X).detach().cpu().numpy()

    if pp.task == "regression":
        pred_scaled = out.reshape(-1, 1)
        return pp.y_scaler.inverse_transform(pred_scaled).ravel()

    return np.argmax(out, axis=1).astype(int)

def metrics_from_prediction(pp: PreparedData, pred: np.ndarray) -> Dict[str, float]:
    if pp.task == "regression":
        return {
            "RMSE": rmse(pp.y_test, pred),
            "R2": float(r2_score(pp.y_test, pred)),
        }

    return {
        "Accuracy": float(accuracy_score(pp.y_test, pred)),
        "F1": float(f1_score(pp.y_test, pred, average="macro", zero_division=0)),
    }

## 7. Baselines

In [8]:
def train_xgboost(pp: PreparedData, seed: int):
    if not HAS_XGBOOST:
        raise RuntimeError("XGBoost is not available.")

    if pp.task == "regression":
        model = XGBRegressor(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=seed,
            n_jobs=-1,
            verbosity=0,
        )

        model.fit(
            pp.X_train,
            pp.y_train_scaled,
            eval_set=[(pp.X_val, pp.y_val_scaled)],
            verbose=False,
        )

        pred = pp.y_scaler.inverse_transform(model.predict(pp.X_test).reshape(-1, 1)).ravel()

    else:
        metric = "logloss" if pp.n_classes == 2 else "mlogloss"

        model = XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=seed,
            n_jobs=-1,
            verbosity=0,
            eval_metric=metric,
        )

        model.fit(
            pp.X_train,
            pp.y_train_scaled,
            eval_set=[(pp.X_val, pp.y_val_scaled)],
            verbose=False,
        )

        pred = model.predict(pp.X_test).astype(int)

    return model, pred

def train_catboost(pp: PreparedData, seed: int):
    if not HAS_CATBOOST:
        raise RuntimeError("CatBoost is not available.")

    if pp.task == "regression":
        model = CatBoostRegressor(
            iterations=400,
            depth=5,
            learning_rate=0.04,
            loss_function="RMSE",
            random_seed=seed,
            verbose=False,
        )

        model.fit(
            pp.X_train,
            pp.y_train_scaled,
            eval_set=(pp.X_val, pp.y_val_scaled),
            verbose=False,
        )

        pred = pp.y_scaler.inverse_transform(model.predict(pp.X_test).reshape(-1, 1)).ravel()

    else:
        loss = "Logloss" if pp.n_classes == 2 else "MultiClass"

        model = CatBoostClassifier(
            iterations=400,
            depth=5,
            learning_rate=0.04,
            loss_function=loss,
            random_seed=seed,
            verbose=False,
        )

        model.fit(
            pp.X_train,
            pp.y_train_scaled,
            eval_set=(pp.X_val, pp.y_val_scaled),
            verbose=False,
        )

        pred = model.predict(pp.X_test).astype(int).ravel()

    return model, pred

def train_ebm(pp: PreparedData, seed: int):
    if not HAS_EBM:
        raise RuntimeError("Interpret EBM is not available.")

    if pp.task == "regression":
        model = ExplainableBoostingRegressor(
            interactions=10,
            random_state=seed,
            n_jobs=-1,
        )

        model.fit(pp.X_train, pp.y_train_scaled)
        pred = pp.y_scaler.inverse_transform(model.predict(pp.X_test).reshape(-1, 1)).ravel()

    else:
        model = ExplainableBoostingClassifier(
            interactions=10,
            random_state=seed,
            n_jobs=-1,
        )

        model.fit(pp.X_train, pp.y_train_scaled)
        pred = model.predict(pp.X_test).astype(int)

    return model, pred

def train_rulefit(pp: PreparedData, seed: int):
    if not HAS_RULEFIT:
        raise RuntimeError("imodels.RuleFit is not available.")

    if pp.task == "regression":
        model = RuleFitRegressor(
            n_estimators=200,
            tree_size=4,
            max_rules=2000,
            random_state=seed,
        )

        model.fit(pp.X_train, pp.y_train_scaled, feature_names=pp.feature_names)
        pred = pp.y_scaler.inverse_transform(model.predict(pp.X_test).reshape(-1, 1)).ravel()

    else:
        base = RuleFitClassifier(
            n_estimators=200,
            tree_size=4,
            max_rules=2000,
            random_state=seed,
        )

        if pp.n_classes == 2:
            model = base
            model.fit(pp.X_train, pp.y_train_scaled, feature_names=pp.feature_names)
            pred = model.predict(pp.X_test).astype(int)

        else:
            model = OneVsRestClassifier(base)
            model.fit(pp.X_train, pp.y_train_scaled)
            pred = model.predict(pp.X_test).astype(int)

    return model, pred

BASELINE_TRAINERS = {
    "EBM": train_ebm,
    "XGBoost": train_xgboost,
    "CatBoost": train_catboost,
    "RuleFit": train_rulefit,
}

## 8. Interpretation utilities

In [9]:
def gradient_saliency(model, pp: PreparedData, n_samples: int = GRADIENT_N_SAMPLES):
    model.eval()

    n = min(n_samples, len(pp.X_train))
    X = torch.tensor(pp.X_train[:n], dtype=torch.float32, device=DEVICE).requires_grad_(True)

    with torch.enable_grad():
        out = model(X)

        score = out.sum()
        score.backward()

    grad = X.grad.detach().cpu().numpy()

    sal = np.abs(grad).mean(axis=0)
    signed = grad.mean(axis=0)

    sal = sal / (sal.max() + 1e-12)

    return sal, signed

def gradient_interaction_matrix(model, pp: PreparedData, n_samples: int = GRADIENT_N_SAMPLES):
    model.eval()

    n = min(n_samples, len(pp.X_train))
    X = torch.tensor(pp.X_train[:n], dtype=torch.float32, device=DEVICE).requires_grad_(True)

    with torch.enable_grad():
        out = model(X)

        score = out.sum()
        grads = torch.autograd.grad(score, X)[0]

    G = grads.detach().cpu().numpy()
    Gc = G - G.mean(axis=0, keepdims=True)

    M = np.abs(Gc.T @ Gc) / max(len(Gc), 1)
    np.fill_diagonal(M, 0.0)

    M = M / (M.max() + 1e-12)
    return M

def rule_mask_matrix(model) -> np.ndarray:
    return torch.sigmoid(model.rules.mask_logit).detach().cpu().numpy()

def top_rule_indices(model, k: int = TOP_RULES_DISPLAY):
    w = model.head.weight.detach().cpu().numpy()
    imp = np.abs(w).sum(axis=0)
    idx = np.argsort(imp)[::-1][:k]
    return idx, imp

def extract_rules_from_margin_model(
    model,
    pp: PreparedData,
    top_k: int = TOP_RULES_DISPLAY,
    max_literals: int = 4,
    absolute_floor: float = RULE_DISPLAY_ABSOLUTE_FLOOR,
    relative_fraction: float = RULE_DISPLAY_RELATIVE_FRACTION,
) -> List[Dict[str, Any]]:
    rules = model.rules

    head_w = model.head.weight.detach().cpu().numpy()
    rule_importance = np.abs(head_w).sum(axis=0)

    mask = torch.sigmoid(rules.mask_logit).detach().cpu().numpy()
    th_scaled = rules.th.detach().cpu().numpy()
    direction = torch.tanh(rules.ineq).detach().cpu().numpy()
    evidence_sign = torch.tanh(rules.esign).detach().cpu().numpy()

    scaler_mean = pp.x_scaler.mean_
    scaler_std = pp.x_scaler.scale_
    th_raw = th_scaled * scaler_std[None, :] + scaler_mean[None, :]

    selected = np.argsort(rule_importance)[::-1][:top_k]
    out = []

    for rank, r in enumerate(selected, start=1):
        cutoff = max(absolute_floor, relative_fraction * float(mask[r].max()))

        active = np.where(mask[r] > cutoff)[0]
        active = active[np.argsort(mask[r, active])[::-1]]
        active = active[:max_literals]

        conditions = []

        for j in active:
            op = ">" if direction[r, j] > 0 else "<"
            ev = "+" if evidence_sign[r, j] > 0 else "-"

            conditions.append(
                {
                    "feature": pp.feature_names[j],
                    "feature_index": int(j),
                    "operator": op,
                    "threshold_raw": float(th_raw[r, j]),
                    "threshold_scaled": float(th_scaled[r, j]),
                    "evidence": ev,
                    "mask": float(mask[r, j]),
                    "text": (
                        f"{pp.feature_names[j]} {op} {th_raw[r, j]:+.3f} "
                        f"(evidence={ev}, mask={mask[r, j]:.2f})"
                    ),
                }
            )

        out.append(
            {
                "rank": rank,
                "rule_id": int(r),
                "importance": float(rule_importance[r]),
                "max_mask": float(mask[r].max()),
                "cutoff": float(cutoff),
                "n_conditions": int(len(conditions)),
                "conditions": conditions,
                "rule_text": " AND ".join([c["text"] for c in conditions])
                if conditions
                else "(no literal above display threshold)",
            }
        )

    return out

def margin_mask_summary(model, top_k: int = TOP_RULES_DISPLAY) -> Dict[str, float]:
    mask = rule_mask_matrix(model)

    entropy = -mask * np.log(mask + 1e-12) - (1.0 - mask) * np.log(1.0 - mask + 1e-12)

    idx, _ = top_rule_indices(model, k=top_k)

    displayed_counts = []
    for r in idx:
        cutoff = max(RULE_DISPLAY_ABSOLUTE_FLOOR, RULE_DISPLAY_RELATIVE_FRACTION * float(mask[r].max()))
        active = np.where(mask[r] > cutoff)[0]
        active = active[np.argsort(mask[r, active])[::-1]]
        active = active[:4]
        displayed_counts.append(len(active))

    out = {
        "MaskMean": float(mask.mean()),
        "MaskMaxMean": float(mask.max(axis=1).mean()),
        "MaskEntropy": float(entropy.mean()),
        "DisplayedTopRuleConditions": int(np.sum(displayed_counts)),
        "MeanDisplayedConditions": float(np.mean(displayed_counts)) if displayed_counts else np.nan,
        "DisplayedRuleSparsity": float(1.0 - np.mean(displayed_counts) / mask.shape[1]) if displayed_counts else np.nan,
    }

    for th in [0.05, 0.10, 0.25, 0.50]:
        active = mask > th
        suffix = str(th).replace(".", "p")
        out[f"StrongLiteralsGT{suffix}"] = int(active.sum())
        out[f"MeanStrongLiteralsPerRuleGT{suffix}"] = float(active.sum(axis=1).mean())
        out[f"RulesWithAnyLiteralGT{suffix}"] = int((active.sum(axis=1) > 0).sum())

    return out

def rulefit_feature_importance(model, pp: PreparedData) -> np.ndarray:
    D = pp.X_train.shape[1]
    fi = np.zeros(D, dtype=float)

    def add_from_rules_df(rules_df):
        if rules_df is None:
            return

        for _, row in rules_df.iterrows():
            text = str(row.get("rule", ""))
            coef = float(abs(row.get("coef", row.get("importance", 0.0))))

            for j, fname in enumerate(pp.feature_names):
                if re.search(rf"\b{fname}\b", text):
                    fi[j] += coef

    try:
        if hasattr(model, "get_rules"):
            add_from_rules_df(model.get_rules())
        elif hasattr(model, "estimators_"):
            for est in model.estimators_:
                if hasattr(est, "get_rules"):
                    add_from_rules_df(est.get_rules())
    except Exception:
        pass

    return fi / (fi.max() + 1e-12)

def baseline_feature_importance(name: str, model, pp: PreparedData) -> np.ndarray:
    D = pp.X_train.shape[1]

    if name in ["XGBoost", "CatBoost"] and hasattr(model, "feature_importances_"):
        fi = np.asarray(model.feature_importances_, dtype=float)[:D]
        return fi / (fi.max() + 1e-12)

    if name == "EBM":
        try:
            imps = np.asarray(model.term_importances(), dtype=float)
            fi = imps[:D]
            return fi / (fi.max() + 1e-12)
        except Exception:
            return np.zeros(D)

    if name == "RuleFit":
        return rulefit_feature_importance(model, pp)

    return np.zeros(D)

def get_rulefit_rule_table(model) -> pd.DataFrame:
    try:
        if hasattr(model, "get_rules"):
            df_rules = model.get_rules()
            cols = [c for c in ["rule", "coef", "support", "importance"] if c in df_rules.columns]
            if cols:
                sort_col = "importance" if "importance" in cols else cols[-1]
                return df_rules[cols].sort_values(sort_col, ascending=False).head(10)
    except Exception:
        pass

    return pd.DataFrame()

def interpret_margin_model(name: str, model, pp: PreparedData) -> Dict[str, Any]:
    fi, signed = gradient_saliency(model, pp)
    inter = gradient_interaction_matrix(model, pp)
    rules = extract_rules_from_margin_model(model, pp)
    mask_summary = margin_mask_summary(model)

    extra = {}

    if name == "SMTE":
        extra["nu"] = F.softplus(model.log_nu).detach().cpu().numpy()
        extra["NuMean"] = float(extra["nu"].mean())
        extra["NuMin"] = float(extra["nu"].min())
        extra["NuMax"] = float(extra["nu"].max())
        extra["template_location"] = model.mu.detach().cpu().numpy()
        extra["template_scale"] = F.softplus(model.log_s).detach().cpu().numpy()

    if name == "GMTE":
        extra["NuMean"] = np.nan
        extra["NuMin"] = np.nan
        extra["NuMax"] = np.nan
        extra["template_location"] = model.loc.detach().cpu().numpy()
        extra["template_scale"] = F.softplus(model.scale).detach().cpu().numpy()

    return {
        "feature_importance": fi,
        "signed_importance": signed,
        "interaction_matrix": inter,
        "rules": rules,
        **mask_summary,
        **extra,
    }

def interpret_baseline_model(name: str, model, pp: PreparedData) -> Dict[str, Any]:
    fi = baseline_feature_importance(name, model, pp)

    rules = []
    if name == "RuleFit":
        rules = get_rulefit_rule_table(model).to_dict("records")

    return {
        "feature_importance": fi,
        "signed_importance": None,
        "interaction_matrix": None,
        "rules": rules,
        "MaskMean": np.nan,
        "MaskMaxMean": np.nan,
        "MaskEntropy": np.nan,
        "DisplayedTopRuleConditions": np.nan,
        "MeanDisplayedConditions": np.nan,
        "DisplayedRuleSparsity": np.nan,
        "NuMean": np.nan,
        "NuMin": np.nan,
        "NuMax": np.nan,
    }

def compute_interpretability_scores(
    ds_name: str,
    pp: PreparedData,
    interp: Dict[str, Any],
    black_box: bool = False,
) -> Dict[str, float]:
    meta = SYNTHETIC_DATASETS[ds_name]
    active = set(meta["active_feats"].keys())
    D = len(pp.feature_names)

    fi = np.asarray(interp.get("feature_importance", np.zeros(D)), dtype=float)
    top = set(np.argsort(fi)[::-1][:meta["n_active"]])

    precision = len(top & active) / max(len(top), 1)
    recall = len(top & active) / max(len(active), 1)
    fsel_f1 = 2 * precision * recall / (precision + recall + 1e-12)

    rules = interp.get("rules", [])

    if rules and all("n_conditions" in r for r in rules):
        avg_conditions = float(np.mean([r["n_conditions"] for r in rules]))
        rule_sparsity = max(0.0, 1.0 - avg_conditions / D)
    elif rules:
        avg_conditions = np.nan
        rule_sparsity = 0.5
    else:
        avg_conditions = np.nan
        rule_sparsity = 0.0 if black_box else 0.5

    # ground-truth interaction candidates are all active-feature pairs.
    M = interp.get("interaction_matrix", None)
    af = sorted(active)
    gt_pairs = set((a, b) for i, a in enumerate(af) for b in af[i + 1:])

    if M is not None and len(gt_pairs) > 0 and np.nanmax(M) > 0:
        pairs = sorted(
            [(M[i, j], i, j) for i in range(D) for j in range(i + 1, D)],
            reverse=True,
        )
        pred_pairs = set((min(i, j), max(i, j)) for _, i, j in pairs[:len(gt_pairs)])
        interaction_score = len(pred_pairs & gt_pairs) / len(gt_pairs)
    else:
        pred_pairs = set()
        interaction_score = 0.5

    signed = interp.get("signed_importance", None)
    correct, total = 0, 0

    if signed is not None:
        for j, known_sign in meta["active_feats"].items():
            if known_sign is not None:
                total += 1
                correct += int(np.sign(signed[j]) == np.sign(known_sign))

    direction_score = correct / total if total > 0 else 0.5

    composite = (
        0.40 * fsel_f1
        + 0.25 * rule_sparsity
        + 0.20 * interaction_score
        + 0.15 * direction_score
    )

    return {
        "FeatSel_F1": float(fsel_f1),
        "FeatSel_Precision": float(precision),
        "FeatSel_Recall": float(recall),
        "RuleSparsity": float(rule_sparsity),
        "MeanDisplayedConditions": float(avg_conditions) if not np.isnan(avg_conditions) else np.nan,
        "InteractDetect": float(interaction_score),
        "Direction": float(direction_score),
        "Composite_I": float(composite),
        "SelectedFeatures": str(sorted(list(top))),
        "PredictedInteractionPairs": str(sorted(list(pred_pairs))),
    }

## 9. Benchmark run

In [10]:
results = []
failed_rows = []

trained_models = {}
interpretations = {}
histories = {}
prepared_data = {}

for seed in SEEDS:
    set_seed(seed)

    for ds_name in SYNTHETIC_DATASETS:
        pp = prepare_dataset(ds_name, seed=seed)
        prepared_data[(seed, ds_name)] = pp

        print("\n" + "=" * 100)
        print(f"Seed={seed} | {ds_name} | task={pp.task}")
        print(f"Ground truth: {SYNTHETIC_DATASETS[ds_name]['true_rule']}")
        print("=" * 100)

        for model_name in ALL_MODELS:
            t0 = now()

            try:
                if model_name in PROPOSED_MODELS:
                    model = build_margin_model(model_name, pp)
                    hist = train_margin_model(model, pp)
                    pred = predict_margin_model(model, pp)

                    metrics = metrics_from_prediction(pp, pred)
                    interp = interpret_margin_model(model_name, model, pp)
                    scores = compute_interpretability_scores(ds_name, pp, interp, black_box=False)

                    params = int(sum(p.numel() for p in model.parameters()))
                    histories[(seed, ds_name, model_name)] = hist

                else:
                    model, pred = BASELINE_TRAINERS[model_name](pp, seed)
                    metrics = metrics_from_prediction(pp, pred)
                    interp = interpret_baseline_model(model_name, model, pp)

                    scores = compute_interpretability_scores(
                        ds_name,
                        pp,
                        interp,
                        black_box=(model_name in ["XGBoost", "CatBoost"]),
                    )

                    params = np.nan

                elapsed = now() - t0

                trained_models[(seed, ds_name, model_name)] = model
                interpretations[(seed, ds_name, model_name)] = interp

                row = {
                    "Seed": seed,
                    "Dataset": ds_name,
                    "Task": pp.task,
                    "Model": model_name,
                    "TrainTime": elapsed,
                    "Params": params,
                    **metrics,
                    **scores,
                    "MaskMean": interp.get("MaskMean", np.nan),
                    "MaskMaxMean": interp.get("MaskMaxMean", np.nan),
                    "MaskEntropy": interp.get("MaskEntropy", np.nan),
                    "DisplayedTopRuleConditions": interp.get("DisplayedTopRuleConditions", np.nan),
                    "MeanDisplayedConditions_FromMask": interp.get("MeanDisplayedConditions", np.nan),
                    "DisplayedRuleSparsity_FromMask": interp.get("DisplayedRuleSparsity", np.nan),
                    "NuMean": interp.get("NuMean", np.nan),
                    "NuMin": interp.get("NuMin", np.nan),
                    "NuMax": interp.get("NuMax", np.nan),
                }

                for key, value in interp.items():
                    if isinstance(value, (int, float, np.integer, np.floating)) and key not in row:
                        row[key] = value

                results.append(row)

                metric_text = (
                    f"RMSE={metrics['RMSE']:.3f} R2={metrics['R2']:.3f}"
                    if pp.task == "regression"
                    else f"Acc={metrics['Accuracy']:.3f} F1={metrics['F1']:.3f}"
                )

                print(
                    f"{model_name:10s} | {metric_text} | "
                    f"I={scores['Composite_I']:.3f} | "
                    f"FeatF1={scores['FeatSel_F1']:.3f} | "
                    f"t={elapsed:.1f}s"
                )

            except Exception as e:
                elapsed = now() - t0

                err = f"{type(e).__name__}: {str(e)[:200]}"
                print(f"{model_name:10s} | FAILED: {err}")

                failed_rows.append(
                    {
                        "Seed": seed,
                        "Dataset": ds_name,
                        "Task": pp.task,
                        "Model": model_name,
                        "TrainTime": elapsed,
                        "Error": err,
                    }
                )

df = pd.DataFrame(results)
failed = pd.DataFrame(failed_rows)

df.to_csv(TABLE_DIR / "table_s2_benchmark_raw_results.csv", index=False)
failed.to_csv(TABLE_DIR / "table_failed_runs.csv", index=False)

display(df.head())
display(failed)


Seed=1 | Hierarchical | task=regression
Ground truth: IF x0>0.5 AND x1<0 THEN +3; IF x2>1 THEN +2*x5; plus 1.5*x8 and 0.8*x0*x1
SMTE       | RMSE=0.642 R2=0.933 | I=0.923 | FeatF1=1.000 | t=13.8s
GMTE       | RMSE=0.698 R2=0.921 | I=0.930 | FeatF1=1.000 | t=3.4s
EBM        | RMSE=0.473 R2=0.964 | I=0.620 | FeatF1=0.800 | t=84.4s
XGBoost    | RMSE=0.632 R2=0.935 | I=0.575 | FeatF1=1.000 | t=0.6s
CatBoost   | RMSE=0.512 R2=0.957 | I=0.575 | FeatF1=1.000 | t=0.9s
RuleFit    | RMSE=0.619 R2=0.938 | I=0.300 | FeatF1=0.000 | t=139.9s

Seed=1 | Friedman1 | task=regression
Ground truth: 10*sin(pi*x0*x1) + 20*(x2-0.5)^2 + 10*x3 + 5*x4
SMTE       | RMSE=0.498 R2=0.988 | I=0.927 | FeatF1=1.000 | t=3.9s
GMTE       | RMSE=0.672 R2=0.978 | I=0.953 | FeatF1=1.000 | t=3.6s
EBM        | RMSE=0.432 R2=0.991 | I=0.700 | FeatF1=1.000 | t=60.6s
XGBoost    | RMSE=0.886 R2=0.963 | I=0.575 | FeatF1=1.000 | t=0.5s
CatBoost   | RMSE=0.537 R2=0.986 | I=0.575 | FeatF1=1.000 | t=0.8s
RuleFit    | RMSE=1.151 R2=0.

,Seed,Dataset,Task,Model,TrainTime,Params,RMSE,R2,FeatSel_F1,FeatSel_Precision,FeatSel_Recall,RuleSparsity,MeanDisplayedConditions,InteractDetect,Direction,Composite_I,SelectedFeatures,PredictedInteractionPairs,MaskMean,MaskMaxMean,MaskEntropy,DisplayedTopRuleConditions,MeanDisplayedConditions_FromMask,DisplayedRuleSparsity_FromMask,NuMean,NuMin,NuMax,DisplayedRuleSparsity,StrongLiteralsGT0p05,MeanStrongLiteralsPerRuleGT0p05,RulesWithAnyLiteralGT0p05,StrongLiteralsGT0p1,MeanStrongLiteralsPerRuleGT0p1,RulesWithAnyLiteralGT0p1,StrongLiteralsGT0p25,MeanStrongLiteralsPerRuleGT0p25,RulesWithAnyLiteralGT0p25,StrongLiteralsGT0p5,MeanStrongLiteralsPerRuleGT0p5,RulesWithAnyLiteralGT0p5,Accuracy,F1
0,1,Hierarchical,regression,SMTE,13.8498,2978.0000,0.6420,0.9329,1.0000,1.0000,1.0000,0.7733,3.4000,0.9000,1.0000,0.9233,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[(0, 1), (0, 2), (0, 5), (0, 8), (1, 2), (1, 5...",0.0504,0.0567,0.1996,17.0000,3.4000,0.7733,1.6849,1.5036,1.8474,0.7733,111.0000,3.4688,32.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN
1,1,Hierarchical,regression,GMTE,3.3880,2946.0000,0.6977,0.9208,1.0000,1.0000,1.0000,0.8000,3.0000,0.9000,1.0000,0.9300,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[(0, 1), (0, 2), (0, 4), (0, 5), (0, 8), (1, 2...",0.0507,0.0624,0.2004,15.0000,3.0000,0.8000,NaN,NaN,NaN,0.8000,84.0000,2.6250,32.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN
2,1,Hierarchical,regression,EBM,84.3540,NaN,0.4731,0.9636,0.8000,0.8000,0.8000,0.5000,NaN,0.5000,0.5000,0.6200,"[np.int64(0), np.int64(1), np.int64(3), np.int...",[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Hierarchical,regression,XGBoost,0.5920,NaN,0.6319,0.9350,1.0000,1.0000,1.0000,0.0000,NaN,0.5000,0.5000,0.5750,"[np.int64(0), np.int64(1), np.int64(2), np.int...",[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,Hierarchical,regression,CatBoost,0.9423,NaN,0.5117,0.9574,1.0000,1.0000,1.0000,0.0000,NaN,0.5000,0.5000,0.5750,"[np.int64(0), np.int64(1), np.int64(2), np.int...",[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


""


## 10. Extracted rule table and feature report

In [11]:
rule_rows = []

for (seed, ds_name, model_name), interp in interpretations.items():
    if model_name not in PROPOSED_MODELS:
        continue

    pp = prepared_data[(seed, ds_name)]

    for r in interp["rules"]:
        rule_rows.append(
            {
                "Seed": seed,
                "Dataset": ds_name,
                "Model": model_name,
                "Rank": r["rank"],
                "RuleID": r["rule_id"],
                "Importance": r["importance"],
                "MaxMask": r["max_mask"],
                "Cutoff": r["cutoff"],
                "NConditions": r["n_conditions"],
                "Rule": r["rule_text"],
            }
        )

rules_df = pd.DataFrame(rule_rows)
rules_df.to_csv(TABLE_DIR / "table_s5_extracted_smte_gmte_rules.csv", index=False)

display(rules_df.head(40))

feature_rows = []

for (seed, ds_name, model_name), interp in interpretations.items():
    pp = prepared_data[(seed, ds_name)]
    active = set(SYNTHETIC_DATASETS[ds_name]["active_feats"].keys())

    fi = np.asarray(interp.get("feature_importance", np.zeros(len(pp.feature_names))), dtype=float)
    order = np.argsort(fi)[::-1][:TOP_FEATURES_REPORT]

    for rank, j in enumerate(order, start=1):
        feature_rows.append(
            {
                "Seed": seed,
                "Dataset": ds_name,
                "Model": model_name,
                "Rank": rank,
                "Feature": pp.feature_names[j],
                "FeatureIndex": int(j),
                "Importance": float(fi[j]),
                "GroundTruth": bool(j in active),
            }
        )

feature_reports = pd.DataFrame(feature_rows)
feature_reports.to_csv(TABLE_DIR / "table_top_features.csv", index=False)

display(feature_reports.head(40))

,Seed,Dataset,Model,Rank,RuleID,Importance,MaxMask,Cutoff,NConditions,Rule
0,1,Hierarchical,SMTE,1,23,0.6168,0.0559,0.0500,4,"x2 > +0.961 (evidence=+, mask=0.06) AND x5 < +..."
1,1,Hierarchical,SMTE,2,17,0.5773,0.0595,0.0500,3,"x2 > +1.330 (evidence=+, mask=0.06) AND x8 < +..."
2,1,Hierarchical,SMTE,3,12,0.5537,0.0593,0.0500,4,"x2 < +0.651 (evidence=-, mask=0.06) AND x0 < -..."
3,1,Hierarchical,SMTE,4,30,0.5504,0.0606,0.0500,2,"x2 < +0.631 (evidence=+, mask=0.06) AND x5 < -..."
4,1,Hierarchical,SMTE,5,31,0.5425,0.0537,0.0500,4,"x2 > +0.835 (evidence=-, mask=0.05) AND x8 < +..."
5,1,Hierarchical,GMTE,1,24,0.6672,0.0665,0.0500,4,"x5 < +0.653 (evidence=+, mask=0.07) AND x2 > +..."
6,1,Hierarchical,GMTE,2,30,0.5605,0.0538,0.0500,3,"x8 > -0.664 (evidence=+, mask=0.05) AND x0 > -..."
7,1,Hierarchical,GMTE,3,15,0.5354,0.0582,0.0500,3,"x5 > +0.953 (evidence=-, mask=0.06) AND x2 < +..."
8,1,Hierarchical,GMTE,4,7,0.5325,0.0631,0.0500,3,"x5 > +0.898 (evidence=+, mask=0.06) AND x2 < +..."
9,1,Hierarchical,GMTE,5,17,0.4508,0.0613,0.0500,2,"x2 > +1.266 (evidence=+, mask=0.06) AND x5 < +..."


,Seed,Dataset,Model,Rank,Feature,FeatureIndex,Importance,GroundTruth
0,1,Hierarchical,SMTE,1,x8,8,1.0000,True
1,1,Hierarchical,SMTE,2,x0,0,0.5551,True
2,1,Hierarchical,SMTE,3,x1,1,0.4937,True
3,1,Hierarchical,SMTE,4,x2,2,0.3190,True
4,1,Hierarchical,SMTE,5,x5,5,0.2621,True
5,1,Hierarchical,SMTE,6,x7,7,0.0334,False
6,1,Hierarchical,SMTE,7,x6,6,0.0316,False
7,1,Hierarchical,SMTE,8,x4,4,0.0309,False
8,1,Hierarchical,GMTE,1,x8,8,1.0000,True
9,1,Hierarchical,GMTE,2,x0,0,0.4731,True


## 11. Summary tables

In [12]:
def normalized_performance_score(sub: pd.DataFrame, task: str) -> pd.DataFrame:
    sub = sub.copy()

    if task == "regression":
        x = sub["RMSE"].astype(float)
        sub["PerfScore"] = 1.0 - (x - x.min()) / (x.max() - x.min() + 1e-12)
        sub["Rank"] = x.rank(ascending=True, method="average")
    else:
        x = sub["F1"].astype(float)
        sub["PerfScore"] = (x - x.min()) / (x.max() - x.min() + 1e-12)
        sub["Rank"] = x.rank(ascending=False, method="average")

    return sub

scored_blocks = []

for (seed, ds_name), pp in prepared_data.items():
    sub = df[(df["Seed"] == seed) & (df["Dataset"] == ds_name)].copy()

    if len(sub) == 0:
        continue

    scored_blocks.append(normalized_performance_score(sub, pp.task))

df_score = pd.concat(scored_blocks, ignore_index=True) if scored_blocks else pd.DataFrame()
df_score.to_csv(TABLE_DIR / "table_ranked_results.csv", index=False)

summary_rows = []

metric_cols = [
    "PerfScore",
    "Rank",
    "TrainTime",
    "Composite_I",
    "FeatSel_F1",
    "FeatSel_Precision",
    "FeatSel_Recall",
    "RuleSparsity",
    "MeanDisplayedConditions",
    "InteractDetect",
    "Direction",
    "RMSE",
    "R2",
    "Accuracy",
    "F1",
    "MaskMean",
    "MaskEntropy",
    "DisplayedTopRuleConditions",
    "MeanDisplayedConditions_FromMask",
    "DisplayedRuleSparsity_FromMask",
    "NuMean",
]

for model_name, g in df_score.groupby("Model"):
    row = {
        "Model": model_name,
        "Runs": len(g),
    }

    for c in metric_cols:
        if c in g.columns:
            vals = pd.to_numeric(g[c], errors="coerce")
            row[f"{c}_mean"] = vals.mean()
            row[f"{c}_std"] = vals.std(ddof=1)
            row[f"{c}_mean_std"] = mean_std_str(vals)
        else:
            row[f"{c}_mean"] = np.nan
            row[f"{c}_std"] = np.nan
            row[f"{c}_mean_std"] = "nan ± nan"

    summary_rows.append(row)

model_summary = pd.DataFrame(summary_rows)

if not model_summary.empty:
    model_summary["PaperScore"] = (
        0.60 * model_summary["PerfScore_mean"]
        + 0.40 * model_summary["Composite_I_mean"]
    )
    model_summary = model_summary.sort_values("PaperScore", ascending=False)

model_summary.to_csv(TABLE_DIR / "table_model_summary.csv", index=False)
display(model_summary)

dataset_summary_rows = []

for (dataset, task, model), g in df_score.groupby(["Dataset", "Task", "Model"]):
    row = {
        "Dataset": dataset,
        "Task": task,
        "Model": model,
        "Runs": len(g),
    }

    for c in metric_cols:
        if c in g.columns:
            vals = pd.to_numeric(g[c], errors="coerce")
            row[f"{c}_mean"] = vals.mean()
            row[f"{c}_std"] = vals.std(ddof=1)
            row[f"{c}_mean_std"] = mean_std_str(vals)
        else:
            row[f"{c}_mean"] = np.nan
            row[f"{c}_std"] = np.nan
            row[f"{c}_mean_std"] = "nan ± nan"

    dataset_summary_rows.append(row)

dataset_summary = pd.DataFrame(dataset_summary_rows)
dataset_summary.to_csv(TABLE_DIR / "table_per_dataset_summary.csv", index=False)

display(dataset_summary.head(30))

paper_rows = []

for _, row in dataset_summary.iterrows():
    if row["Task"] == "regression":
        metric_name = "RMSE"
        performance = row.get("RMSE_mean_std", "")
    else:
        metric_name = "Macro-F1"
        performance = row.get("F1_mean_std", "")

    paper_rows.append(
        {
            "Dataset": row["Dataset"],
            "Task": row["Task"],
            "Model": row["Model"],
            "Metric": metric_name,
            "Performance": performance,
            "FeatureRecovery_F1": row.get("FeatSel_F1_mean_std", ""),
            "InteractionDetection": row.get("InteractDetect_mean_std", ""),
            "DisplayedRuleSparsity": row.get("RuleSparsity_mean_std", ""),
            "CompositeInterpretability": row.get("Composite_I_mean_std", ""),
            "TrainTimeSec": row.get("TrainTime_mean_std", ""),
            "NuMean": row.get("NuMean_mean_std", ""),
        }
    )

paper_table = pd.DataFrame(paper_rows)
paper_table.to_csv(TABLE_DIR / "paper_table_main_synthetic_benchmark.csv", index=False)

display(paper_table.head(40))

,Model,Runs,PerfScore_mean,PerfScore_std,PerfScore_mean_std,Rank_mean,Rank_std,Rank_mean_std,TrainTime_mean,TrainTime_std,TrainTime_mean_std,Composite_I_mean,Composite_I_std,Composite_I_mean_std,FeatSel_F1_mean,FeatSel_F1_std,FeatSel_F1_mean_std,FeatSel_Precision_mean,FeatSel_Precision_std,FeatSel_Precision_mean_std,FeatSel_Recall_mean,FeatSel_Recall_std,FeatSel_Recall_mean_std,RuleSparsity_mean,RuleSparsity_std,RuleSparsity_mean_std,MeanDisplayedConditions_mean,MeanDisplayedConditions_std,MeanDisplayedConditions_mean_std,InteractDetect_mean,InteractDetect_std,InteractDetect_mean_std,Direction_mean,Direction_std,Direction_mean_std,RMSE_mean,RMSE_std,RMSE_mean_std,R2_mean,R2_std,R2_mean_std,Accuracy_mean,Accuracy_std,Accuracy_mean_std,F1_mean,F1_std,F1_mean_std,MaskMean_mean,MaskMean_std,MaskMean_mean_std,MaskEntropy_mean,MaskEntropy_std,MaskEntropy_mean_std,DisplayedTopRuleConditions_mean,DisplayedTopRuleConditions_std,DisplayedTopRuleConditions_mean_std,MeanDisplayedConditions_FromMask_mean,MeanDisplayedConditions_FromMask_std,MeanDisplayedConditions_FromMask_mean_std,DisplayedRuleSparsity_FromMask_mean,DisplayedRuleSparsity_FromMask_std,DisplayedRuleSparsity_FromMask_mean_std,NuMean_mean,NuMean_std,NuMean_mean_std,PaperScore
1,EBM,25,0.9093,0.1179,0.909 ± 0.118,2.3000,1.1637,2.300 ± 1.164,77.9457,79.8800,77.946 ± 79.880,0.6480,0.0748,0.648 ± 0.075,0.8700,0.1871,0.870 ± 0.187,0.8700,0.1871,0.870 ± 0.187,0.8700,0.1871,0.870 ± 0.187,0.5000,0.0000,0.500 ± 0.000,NaN,NaN,nan ± nan,0.5000,0.0000,0.500 ± 0.000,0.5000,0.0000,0.500 ± 0.000,0.4766,0.0370,0.477 ± 0.037,0.9732,0.0185,0.973 ± 0.018,0.9938,0.0081,0.994 ± 0.008,0.9944,0.0067,0.994 ± 0.007,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,0.8048
0,CatBoost,25,0.8940,0.1617,0.894 ± 0.162,2.4400,0.9929,2.440 ± 0.993,1.2242,0.4844,1.224 ± 0.484,0.5750,0.0000,0.575 ± 0.000,1.0000,0.0000,1.000 ± 0.000,1.0000,0.0000,1.000 ± 0.000,1.0000,0.0000,1.000 ± 0.000,0.0000,0.0000,0.000 ± 0.000,NaN,NaN,nan ± nan,0.5000,0.0000,0.500 ± 0.000,0.5000,0.0000,0.500 ± 0.000,0.5205,0.0409,0.521 ± 0.041,0.9716,0.0154,0.972 ± 0.015,0.9963,0.0030,0.996 ± 0.003,0.9951,0.0047,0.995 ± 0.005,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,0.7664
5,XGBoost,25,0.6087,0.3620,0.609 ± 0.362,3.4400,1.6286,3.440 ± 1.629,0.5458,0.2076,0.546 ± 0.208,0.5750,0.0000,0.575 ± 0.000,1.0000,0.0000,1.000 ± 0.000,1.0000,0.0000,1.000 ± 0.000,1.0000,0.0000,1.000 ± 0.000,0.0000,0.0000,0.000 ± 0.000,NaN,NaN,nan ± nan,0.5000,0.0000,0.500 ± 0.000,0.5000,0.0000,0.500 ± 0.000,0.7633,0.1537,0.763 ± 0.154,0.9468,0.0174,0.947 ± 0.017,0.9953,0.0053,0.995 ± 0.005,0.9945,0.0068,0.994 ± 0.007,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,0.5952
2,GMTE,25,0.3883,0.3878,0.388 ± 0.388,4.6800,1.2819,4.680 ± 1.282,3.4507,0.0849,3.451 ± 0.085,0.8864,0.0532,0.886 ± 0.053,1.0000,0.0000,1.000 ± 0.000,1.0000,0.0000,1.000 ± 0.000,1.0000,0.0000,1.000 ± 0.000,0.8501,0.0343,0.850 ± 0.034,2.2480,0.5141,2.248 ± 0.514,0.8893,0.1170,0.889 ± 0.117,0.6400,0.3686,0.640 ± 0.369,0.6879,0.0746,0.688 ± 0.075,0.9457,0.0353,0.946 ± 0.035,0.9838,0.0125,0.984 ± 0.012,0.9830,0.0124,0.983 ± 0.012,0.0502,0.0003,0.050 ± 0.000,0.1991,0.0008,0.199 ± 0.001,11.2400,2.5703,11.240 ± 2.570,2.2480,0.5141,2.248 ± 0.514,0.8501,0.0343,0.850 ± 0.034,NaN,NaN,nan ± nan,0.5876
4,SMTE,25,0.3649,0.3710,0.365 ± 0.371,4.4400,1.5297,4.440 ± 1.530,4.2836,1.9962,4.284 ± 1.996,0.8651,0.0635,0.865 ± 0.064,0.9900,0.0500,0.990 ± 0.050,0.9900,0.0500,0.990 ± 0.050,0.9900,0.0500,0.990 ± 0.050,0.8075,0.0444,0.807 ± 0.044,2.8880,0.6660,2.888 ± 0.666,0.7960,0.1733,0.796 ± 0.173,0.7200,0.3253,0.720 ± 0.325,0.5703,0.0773,0.570 ± 0.077,0.9583,0.0317,0.958 ± 0.032,0.9797,0.0157,0.980 ± 0.016,0.9790,0.0140,0.979 ± 0.014,0.0502,0.0002,0.050 ± 0.000,0.1990,0.0007,0.199 ± 0.001,14.4400,3.3302,14.440 ± 3.330,2.8880,0.6660,2.888 ± 0.6

,Dataset,Task,Model,Runs,PerfScore_mean,PerfScore_std,PerfScore_mean_std,Rank_mean,Rank_std,Rank_mean_std,TrainTime_mean,TrainTime_std,TrainTime_mean_std,Composite_I_mean,Composite_I_std,Composite_I_mean_std,FeatSel_F1_mean,FeatSel_F1_std,FeatSel_F1_mean_std,FeatSel_Precision_mean,FeatSel_Precision_std,FeatSel_Precision_mean_std,FeatSel_Recall_mean,FeatSel_Recall_std,FeatSel_Recall_mean_std,RuleSparsity_mean,RuleSparsity_std,RuleSparsity_mean_std,MeanDisplayedConditions_mean,MeanDisplayedConditions_std,MeanDisplayedConditions_mean_std,InteractDetect_mean,InteractDetect_std,InteractDetect_mean_std,Direction_mean,Direction_std,Direction_mean_std,RMSE_mean,RMSE_std,RMSE_mean_std,R2_mean,R2_std,R2_mean_std,Accuracy_mean,Accuracy_std,Accuracy_mean_std,F1_mean,F1_std,F1_mean_std,MaskMean_mean,MaskMean_std,MaskMean_mean_std,MaskEntropy_mean,MaskEntropy_std,MaskEntropy_mean_std,DisplayedTopRuleConditions_mean,DisplayedTopRuleConditions_std,DisplayedTopRuleConditions_mean_std,MeanDisplayedConditions_FromMask_mean,MeanDisplayedConditions_FromMask_std,MeanDisplayedConditions_FromMask_mean_std,DisplayedRuleSparsity_FromMask_mean,DisplayedRuleSparsity_FromMask_std,DisplayedRuleSparsity_FromMask_mean_std,NuMean_mean,NuMean_std,NuMean_mean_std
0,Friedman1,regression,CatBoost,5,0.8677,0.0467,0.868 ± 0.047,3.0000,0.7071,3.000 ± 0.707,0.7599,0.0257,0.760 ± 0.026,0.5750,0.0000,0.575 ± 0.000,1.0000,0.0000,1.000 ± 0.000,1.0000,0.0000,1.000 ± 0.000,1.0000,0.0000,1.000 ± 0.000,0.0000,0.0000,0.000 ± 0.000,NaN,NaN,nan ± nan,0.5000,0.0000,0.500 ± 0.000,0.5000,0.0000,0.500 ± 0.000,0.5512,0.0308,0.551 ± 0.031,0.9860,0.0017,0.986 ± 0.002,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan
1,Friedman1,regression,EBM,5,0.9878,0.0272,0.988 ± 0.027,1.2000,0.4472,1.200 ± 0.447,79.7141,11.8097,79.714 ± 11.810,0.7000,0.0000,0.700 ± 0.000,1.0000,0.0000,1.000 ± 0.000,1.0000,0.0000,1.000 ± 0.000,1.0000,0.0000,1.000 ± 0.000,0.5000,0.0000,0.500 ± 0.000,NaN,NaN,nan ± nan,0.5000,0.0000,0.500 ± 0.000,0.5000,0.0000,0.500 ± 0.000,0.4566,0.0371,0.457 ± 0.037,0.9904,0.0016,0.990 ± 0.002,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan
2,Friedman1,regression,GMTE,5,0.7158,0.1368,0.716 ± 0.137,3.8000,0.4472,3.800 ± 0.447,3.4832,0.0952,3.483 ± 0.095,0.9447,0.0102,0.945 ± 0.010,1.0000,0.0000,1.000 ± 0.000,1.0000,0.0000,1.000 ± 0.000,1.0000,0.0000,1.000 ± 0.000,0.8267,0.0163,0.827 ± 0.016,2.6000,0.2449,2.600 ± 0.245,0.9400,0.0548,0.940 ± 0.055,1.0000,0.0000,1.000 ± 0.000,0.6771,0.1096,0.677 ± 0.110,0.9785,0.0072,0.978 ± 0.007,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,0.0502,0.0002,0.050 ± 0.000,0.1989,0.0007,0.199 ± 0.001,13.0000,1.2247,13.000 ± 1.225,2.6000,0.2449,2.600 ± 0.245,0.8267,0.0163,0.827 ± 0.016,NaN,NaN,nan ± nan
3,Friedman1,regression,RuleFit,5,0.0000,0.0000,0.000 ± 0.000,6.0000,0.0000,6.000 ± 0.000,125.7787,10.4359,125.779 ± 10.436,0.3000,0.0000,0.300 ± 0.000,0.0000,0.0000,0.000 ± 0.000,0.0000,0.0000,0.000 ± 0.000,0.0000,0.0000,0.000 ± 0.000,0.5000,0.0000,0.500 ± 0.000,NaN,NaN,nan ± nan,0.5000,0.0000,0.500 ± 0.000,0.5000,0.0000,0.500 ± 0.000,1.2573,0.1321,1.257 ± 0.132,0.9267,0.0162,0.927 ± 0.016,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan
4,Friedman1,regression,SMTE,5,0.9250,0.0727,0.925 ± 0.073,2.0000,0.7071,2.000 ± 0.707,3.8763,0.0710,3.876 ± 0.071,0.9107,0.0285,0.911 ± 0.029,1.0000,0.0000,1.000 ± 0.000,1.0000,0.0000,1.000 ± 0.000,1.0000,0.0000,1.000 ± 0.000,0.7547,0.0292,0.755 ± 0.029,3.6800,0.4382,3.680 ± 0.438,0.8600,0.1140,0.860 ± 0.114,1.0000,0.0000,1.000 ± 0.000,0.5077,0.0567,0.508 ± 0.057,0.9880,0.0028,0.988 ± 0.003,NaN,NaN,nan ± nan,NaN,NaN,nan ± nan,0.0501,0.0003,0.050 ± 0.000,0.1988,0.0008,0.199 ± 0.001,18.4000,2.1909,18.400 ± 2.191,3.6800,0.4382,3.680 ± 0.438,0.7547,0.0292,0.7

,Dataset,Task,Model,Metric,Performance,FeatureRecovery_F1,InteractionDetection,DisplayedRuleSparsity,CompositeInterpretability,TrainTimeSec,NuMean
0,Friedman1,regression,CatBoost,RMSE,0.551 ± 0.031,1.000 ± 0.000,0.500 ± 0.000,0.000 ± 0.000,0.575 ± 0.000,0.760 ± 0.026,nan ± nan
1,Friedman1,regression,EBM,RMSE,0.457 ± 0.037,1.000 ± 0.000,0.500 ± 0.000,0.500 ± 0.000,0.700 ± 0.000,79.714 ± 11.810,nan ± nan
2,Friedman1,regression,GMTE,RMSE,0.677 ± 0.110,1.000 ± 0.000,0.940 ± 0.055,0.827 ± 0.016,0.945 ± 0.010,3.483 ± 0.095,nan ± nan
3,Friedman1,regression,RuleFit,RMSE,1.257 ± 0.132,0.000 ± 0.000,0.500 ± 0.000,0.500 ± 0.000,0.300 ± 0.000,125.779 ± 10.436,nan ± nan
4,Friedman1,regression,SMTE,RMSE,0.508 ± 0.057,1.000 ± 0.000,0.860 ± 0.114,0.755 ± 0.029,0.911 ± 0.029,3.876 ± 0.071,1.750 ± 0.016
5,Friedman1,regression,XGBoost,RMSE,0.907 ± 0.019,1.000 ± 0.000,0.500 ± 0.000,0.000 ± 0.000,0.575 ± 0.000,0.502 ± 0.014,nan ± nan
6,Hierarchical,regression,CatBoost,RMSE,0.490 ± 0.022,1.000 ± 0.000,0.500 ± 0.000,0.000 ± 0.000,0.575 ± 0.000,0.810 ± 0.077,nan ± nan
7,Hierarchical,regression,EBM,RMSE,0.496 ± 0.027,0.800 ± 0.000,0.500 ± 0.000,0.500 ± 0.000,0.620 ± 0.000,63.863 ± 12.666,nan ± nan
8,Hierarchical,regression,GMTE,RMSE,0.699 ± 0.015,1.000 ± 0.000,0.840 ± 0.089,0.816 ± 0.026,0.922 ± 0.016,3.443 ± 0.040,nan ± nan
9,Hierarchical,regression,RuleFit,RMSE,0.626 ± 0.051,0.000 ± 0.000,0.500 ± 0.000,0.500 ± 0.000,0.300 ± 0.000,159.102 ± 33.018,nan ± nan


## 12. Per-dataset winners

In [13]:
winner_rows = []

for (seed, ds_name), pp in prepared_data.items():
    sub = df_score[(df_score["Seed"] == seed) & (df_score["Dataset"] == ds_name)].copy()

    if sub.empty:
        continue

    if pp.task == "regression":
        best_idx = sub["RMSE"].idxmin()
        metric = "RMSE"
        value = sub.loc[best_idx, "RMSE"]
    else:
        best_idx = sub["F1"].idxmax()
        metric = "Macro-F1"
        value = sub.loc[best_idx, "F1"]

    winner_rows.append(
        {
            "Seed": seed,
            "Dataset": ds_name,
            "Task": pp.task,
            "Winner": sub.loc[best_idx, "Model"],
            "Metric": metric,
            "Value": value,
        }
    )

winner_table = pd.DataFrame(winner_rows)
winner_table.to_csv(TABLE_DIR / "table_per_dataset_winners.csv", index=False)
display(winner_table)

,Seed,Dataset,Task,Winner,Metric,Value
0,1,Hierarchical,regression,EBM,RMSE,0.4731
1,1,Friedman1,regression,EBM,RMSE,0.4322
2,1,XorChain,classification,CatBoost,Macro-F1,1.0000
3,1,ThresholdCls,classification,EBM,Macro-F1,0.9975
4,1,MultiThresh,classification,SMTE,Macro-F1,1.0000
5,12,Hierarchical,regression,CatBoost,RMSE,0.5068
6,12,Friedman1,regression,SMTE,RMSE,0.4598
7,12,XorChain,classification,XGBoost,Macro-F1,1.0000
8,12,ThresholdCls,classification,XGBoost,Macro-F1,0.9975
9,12,MultiThresh,classification,GMTE,Macro-F1,1.0000


## 13. Paper-style figures

In [14]:
if not model_summary.empty:
    fig, ax = plt.subplots(figsize=(6.8, 4.8))

    plot_df = model_summary.copy()

    t = plot_df["TrainTime_mean"].astype(float)
    time_scaled = 80 + 320 * (t - t.min()) / (t.max() - t.min() + 1e-12)

    for i, row in plot_df.iterrows():
        m = row["Model"]
        ax.scatter(
            row["Composite_I_mean"],
            row["PerfScore_mean"],
            s=float(time_scaled.loc[i]),
            color=MODEL_COLORS.get(m, "gray"),
            edgecolor="black",
            linewidth=0.7,
            alpha=0.85,
        )
        ax.text(row["Composite_I_mean"] + 0.006, row["PerfScore_mean"] + 0.006, m)

    ax.set_xlabel("Interpretability score")
    ax.set_ylabel("Dataset-normalized predictive performance")
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)

    legend_handles = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            label=m,
            markerfacecolor=MODEL_COLORS.get(m, "gray"),
            markeredgecolor="black",
            markersize=8,
        )
        for m in ALL_MODELS
    ]
    ax.legend(handles=legend_handles, frameon=True, loc="lower left")

    savefig("figure_1_accuracy_interpretability_frontier")

In [15]:
if not df_score.empty:
    heat = df_score.pivot_table(
        index="Model",
        columns="Dataset",
        values="PerfScore",
        aggfunc="mean",
    )

    order = [m for m in model_summary["Model"].tolist() if m in heat.index]
    heat = heat.reindex(order)

    fig, ax = plt.subplots(figsize=(8.2, 3.8))

    sns.heatmap(
        heat,
        cmap="viridis",
        vmin=0,
        vmax=1,
        annot=True,
        fmt=".2f",
        linewidths=0.5,
        cbar_kws={"label": "normalized performance"},
        ax=ax,
    )

    ax.set_xlabel("")
    ax.set_ylabel("")

    savefig("figure_2_performance_heatmap")

In [16]:
if not df_score.empty:
    decomp = (
        df_score.groupby("Model")[["FeatSel_F1", "RuleSparsity", "InteractDetect", "Direction"]]
        .mean()
    )

    order = [m for m in model_summary["Model"].tolist() if m in decomp.index]
    decomp = decomp.reindex(order)

    fig, ax = plt.subplots(figsize=(8.4, 4.2))

    bottom = np.zeros(len(decomp))

    components = [
        ("FeatSel_F1", "#2563EB"),
        ("RuleSparsity", "#059669"),
        ("InteractDetect", "#D97706"),
        ("Direction", "#7C3AED"),
    ]

    for col, color in components:
        vals = decomp[col].values
        ax.bar(decomp.index, vals, bottom=bottom, label=col, color=color, alpha=0.88)
        bottom += vals

    ax.set_ylabel("sum of component scores")
    ax.legend(ncol=2, frameon=True)
    ax.tick_params(axis="x", rotation=30)

    savefig("figure_3_interpretability_decomposition")

In [17]:
if not df_score.empty:
    feat_heat = df_score.pivot_table(
        index="Model",
        columns="Dataset",
        values="FeatSel_F1",
        aggfunc="mean",
    )

    order = [m for m in model_summary["Model"].tolist() if m in feat_heat.index]
    feat_heat = feat_heat.reindex(order)

    fig, ax = plt.subplots(figsize=(8.2, 3.8))

    sns.heatmap(
        feat_heat,
        cmap="mako",
        vmin=0,
        vmax=1,
        annot=True,
        fmt=".2f",
        linewidths=0.5,
        cbar_kws={"label": "feature-selection F1"},
        ax=ax,
    )

    ax.set_xlabel("")
    ax.set_ylabel("")

    savefig("figure_4_feature_recovery_heatmap")

In [18]:
if not model_summary.empty:
    radar_metrics = ["PerfScore_mean", "Composite_I_mean", "FeatSel_F1_mean", "RuleSparsity_mean"]

    radar_df = model_summary.set_index("Model")[radar_metrics].reindex(ALL_MODELS).fillna(0)

    labels = ["Performance", "Interpretability", "Feature recovery", "Rule sparsity"]

    angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
    angles += angles[:1]

    fig = plt.figure(figsize=(6.2, 6.2))
    ax = plt.subplot(111, polar=True)

    for model_name in radar_df.index:
        vals = radar_df.loc[model_name].values.tolist()
        vals += vals[:1]

        ax.plot(
            angles,
            vals,
            label=model_name,
            color=MODEL_COLORS.get(model_name, "gray"),
            linewidth=2,
        )
        ax.fill(
            angles,
            vals,
            color=MODEL_COLORS.get(model_name, "gray"),
            alpha=0.08,
        )

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels)
    ax.set_yticks([0.25, 0.50, 0.75, 1.00])
    ax.set_ylim(0, 1)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.10), frameon=True)

    savefig("figure_5_model_card_radar")

## 14. Learned structure plots for SMTE / GMTE

In [19]:
def plot_proposed_structure(seed: int = SEED, ds_name: str = "Hierarchical", top_k: int = 8):
    pp = prepared_data[(seed, ds_name)]

    fig, axes = plt.subplots(2, 3, figsize=(12, 6.2))

    for row_idx, model_name in enumerate(["SMTE", "GMTE"]):
        key = (seed, ds_name, model_name)

        if key not in trained_models:
            continue

        model = trained_models[key]
        idx, imp = top_rule_indices(model, top_k)

        mask = rule_mask_matrix(model)[idx]

        mask_vmin = float(np.min(mask))
        mask_vmax = float(np.max(mask))

        if mask_vmax - mask_vmin < 1e-4:
            mask_vmax = mask_vmin + 1e-4

        sns.heatmap(
            mask,
            cmap="Blues",
            vmin=mask_vmin,
            vmax=mask_vmax,
            xticklabels=pp.feature_names,
            yticklabels=[f"r{r}" for r in idx],
            ax=axes[row_idx, 0],
            cbar=row_idx == 0,
        )

        axes[row_idx, 0].set_title(f"{model_name}: literal masks")
        axes[row_idx, 0].set_xlabel("")
        axes[row_idx, 0].set_ylabel("top rules")
        axes[row_idx, 0].tick_params(axis="x", rotation=90, labelsize=7)

        if model_name == "SMTE":
            loc = model.mu.detach().cpu().numpy()[idx]
            scale = F.softplus(model.log_s).detach().cpu().numpy()[idx]
            loc_title = "template location"
            scale_title = "template scale"
        else:
            loc = model.loc.detach().cpu().numpy()[idx]
            scale = F.softplus(model.scale).detach().cpu().numpy()[idx]
            loc_title = "template location"
            scale_title = "template scale"

        sns.heatmap(
            loc,
            cmap="coolwarm",
            center=0,
            xticklabels=pp.feature_names,
            yticklabels=False,
            ax=axes[row_idx, 1],
            cbar=row_idx == 0,
        )

        axes[row_idx, 1].set_title(f"{model_name}: {loc_title}")
        axes[row_idx, 1].set_xlabel("")
        axes[row_idx, 1].tick_params(axis="x", rotation=90, labelsize=7)

        sns.heatmap(
            scale,
            cmap="YlGnBu",
            xticklabels=pp.feature_names,
            yticklabels=False,
            ax=axes[row_idx, 2],
            cbar=row_idx == 0,
        )

        axes[row_idx, 2].set_title(f"{model_name}: {scale_title}")
        axes[row_idx, 2].set_xlabel("")
        axes[row_idx, 2].tick_params(axis="x", rotation=90, labelsize=7)

    savefig(f"figure_6_rule_structure_{ds_name}")

primary_seed = SEEDS[0]

if (primary_seed, "Hierarchical", "SMTE") in trained_models:
    plot_proposed_structure(seed=primary_seed, ds_name="Hierarchical", top_k=8)

## 15. Training curves

In [20]:
fig, axes = plt.subplots(1, 2, figsize=(10.2, 3.8), sharey=False)

for ax, ds_name in zip(axes, ["Hierarchical", "ThresholdCls"]):
    for model_name in ["SMTE", "GMTE"]:
        hist = histories.get((primary_seed, ds_name, model_name), None)

        if hist is None:
            continue

        ax.plot(
            hist["val_loss"],
            label=model_name,
            color=MODEL_COLORS[model_name],
            linewidth=2,
        )

    ax.set_title(ds_name)
    ax.set_xlabel("epoch")
    ax.set_ylabel("validation loss")
    ax.legend(frameon=True)

savefig("figure_7_training_curves")

## 16. Rule and explanation report

In [21]:
def print_margin_rules(seed: int, ds_name: str, model_name: str, top_k: int = TOP_RULES_DISPLAY):
    key = (seed, ds_name, model_name)

    if key not in interpretations:
        print(f"No interpretation found for {key}")
        return

    interp = interpretations[key]

    print(f"\n{model_name} rules on {ds_name} | seed={seed}")
    print("-" * 90)
    print("Ground truth:", SYNTHETIC_DATASETS[ds_name]["true_rule"])

    for r in interp["rules"][:top_k]:
        print(
            f"\nRule {r['rule_id']} | "
            f"rank={r['rank']} | "
            f"importance={r['importance']:.3f} | "
            f"conditions={r['n_conditions']} | "
            f"cutoff={r['cutoff']:.3f}"
        )

        for c in r["conditions"]:
            print("  -", c["text"])

for ds_name in ["Hierarchical", "ThresholdCls"]:
    for model_name in ["SMTE", "GMTE"]:
        print_margin_rules(primary_seed, ds_name, model_name)


SMTE rules on Hierarchical | seed=1
------------------------------------------------------------------------------------------
Ground truth: IF x0>0.5 AND x1<0 THEN +3; IF x2>1 THEN +2*x5; plus 1.5*x8 and 0.8*x0*x1

Rule 23 | rank=1 | importance=0.617 | conditions=4 | cutoff=0.050
  - x2 > +0.961 (evidence=+, mask=0.06)
  - x5 < +0.749 (evidence=-, mask=0.05)
  - x8 > +0.552 (evidence=-, mask=0.05)
  - x0 < +0.444 (evidence=+, mask=0.05)

Rule 17 | rank=2 | importance=0.577 | conditions=3 | cutoff=0.050
  - x2 > +1.330 (evidence=+, mask=0.06)
  - x8 < +1.583 (evidence=+, mask=0.05)
  - x5 > +1.268 (evidence=+, mask=0.05)

Rule 12 | rank=3 | importance=0.554 | conditions=4 | cutoff=0.050
  - x2 < +0.651 (evidence=-, mask=0.06)
  - x0 < -0.172 (evidence=+, mask=0.05)
  - x5 < -0.809 (evidence=+, mask=0.05)
  - x9 > -0.389 (evidence=+, mask=0.05)

Rule 30 | rank=4 | importance=0.550 | conditions=2 | cutoff=0.050
  - x2 < +0.631 (evidence=+, mask=0.06)
  - x5 < -0.209 (evidence=+, mask=0.

## 17. Top-feature report

In [22]:
display(feature_reports.head(50))

,Seed,Dataset,Model,Rank,Feature,FeatureIndex,Importance,GroundTruth
0,1,Hierarchical,SMTE,1,x8,8,1.0000,True
1,1,Hierarchical,SMTE,2,x0,0,0.5551,True
2,1,Hierarchical,SMTE,3,x1,1,0.4937,True
3,1,Hierarchical,SMTE,4,x2,2,0.3190,True
4,1,Hierarchical,SMTE,5,x5,5,0.2621,True
5,1,Hierarchical,SMTE,6,x7,7,0.0334,False
6,1,Hierarchical,SMTE,7,x6,6,0.0316,False
7,1,Hierarchical,SMTE,8,x4,4,0.0309,False
8,1,Hierarchical,GMTE,1,x8,8,1.0000,True
9,1,Hierarchical,GMTE,2,x0,0,0.4731,True


## 18. Hyperparameter table

In [23]:
hyper_rows = [
    {"model": "SMTE", "hyperparameter": "template", "value": "Student-t signed log-margin", "selection": "fixed"},
    {"model": "GMTE", "hyperparameter": "template", "value": "Gaussian signed log-margin", "selection": "fixed"},
    {"model": "SMTE/GMTE", "hyperparameter": "n_rules", "value": N_RULES, "selection": "fixed from simple synthetic notebook"},
    {"model": "SMTE/GMTE", "hyperparameter": "epochs", "value": EPOCHS, "selection": "fixed with early stopping"},
    {"model": "SMTE/GMTE", "hyperparameter": "patience", "value": PATIENCE, "selection": "fixed"},
    {"model": "SMTE/GMTE", "hyperparameter": "batch_size", "value": BATCH_SIZE, "selection": "fixed"},
    {"model": "SMTE/GMTE", "hyperparameter": "learning_rate", "value": LR, "selection": "fixed"},
    {"model": "SMTE/GMTE", "hyperparameter": "weight_decay", "value": WEIGHT_DECAY, "selection": "fixed"},
    {"model": "SMTE/GMTE", "hyperparameter": "l1_mask", "value": L1_MASK, "selection": "fixed; no sparse-mask tuning"},
    {"model": "SMTE/GMTE", "hyperparameter": "rule_beta", "value": RULE_BETA, "selection": "fixed"},
    {"model": "EBM", "hyperparameter": "interactions", "value": 10, "selection": "fixed"},
    {"model": "XGBoost", "hyperparameter": "n_estimators / max_depth", "value": "300 / 5", "selection": "fixed"},
    {"model": "CatBoost", "hyperparameter": "iterations / depth", "value": "400 / 5", "selection": "fixed"},
    {"model": "RuleFit", "hyperparameter": "n_estimators / tree_size / max_rules", "value": "200 / 4 / 2000", "selection": "fixed"},
]

hyper_table = pd.DataFrame(hyper_rows)
hyper_table.to_csv(TABLE_DIR / "table_hyperparameters_synthetic.csv", index=False)
display(hyper_table)

,model,hyperparameter,value,selection
0,SMTE,template,Student-t signed log-margin,fixed
1,GMTE,template,Gaussian signed log-margin,fixed
2,SMTE/GMTE,n_rules,32,fixed from simple synthetic notebook
3,SMTE/GMTE,epochs,150,fixed with early stopping
4,SMTE/GMTE,patience,20,fixed
5,SMTE/GMTE,batch_size,256,fixed
6,SMTE/GMTE,learning_rate,0.0030,fixed
7,SMTE/GMTE,weight_decay,0.0001,fixed
8,SMTE/GMTE,l1_mask,0.0001,fixed; no sparse-mask tuning
9,SMTE/GMTE,rule_beta,6.0000,fixed


## 19. Compact summary

In [24]:
print("\n" + "=" * 90)
print("SYNTHETIC BENCHMARK SUMMARY")
print("=" * 90)

if not model_summary.empty:
    best_overall = model_summary.iloc[0]["Model"]
    best_perf = model_summary.sort_values("PerfScore_mean", ascending=False).iloc[0]["Model"]
    best_interp = model_summary.sort_values("Composite_I_mean", ascending=False).iloc[0]["Model"]

    print(f"Best overall paper score      : {best_overall}")
    print(f"Best average performance      : {best_perf}")
    print(f"Best average interpretability : {best_interp}")

print("\nPer-dataset winners:")
for _, row in winner_table.iterrows():
    print(
        f"  seed={int(row['Seed'])} | "
        f"{row['Dataset']:18s} -> {row['Winner']:10s} "
        f"{row['Metric']}={row['Value']:.4f}"
    )

print("\nFailed runs:")
print(len(failed))


SYNTHETIC BENCHMARK SUMMARY
Best overall paper score      : EBM
Best average performance      : EBM
Best average interpretability : GMTE

Per-dataset winners:
  seed=1 | Hierarchical       -> EBM        RMSE=0.4731
  seed=1 | Friedman1          -> EBM        RMSE=0.4322
  seed=1 | XorChain           -> CatBoost   Macro-F1=1.0000
  seed=1 | ThresholdCls       -> EBM        Macro-F1=0.9975
  seed=1 | MultiThresh        -> SMTE       Macro-F1=1.0000
  seed=12 | Hierarchical       -> CatBoost   RMSE=0.5068
  seed=12 | Friedman1          -> SMTE       RMSE=0.4598
  seed=12 | XorChain           -> XGBoost    Macro-F1=1.0000
  seed=12 | ThresholdCls       -> XGBoost    Macro-F1=0.9975
  seed=12 | MultiThresh        -> GMTE       Macro-F1=1.0000
  seed=123 | Hierarchical       -> EBM        RMSE=0.4680
  seed=123 | Friedman1          -> EBM        RMSE=0.4827
  seed=123 | XorChain           -> RuleFit    Macro-F1=1.0000
  seed=123 | ThresholdCls       -> CatBoost   Macro-F1=0.9975
  seed=123 

## 20. Manifest

In [25]:
manifest = {
    "output_dir": str(OUTPUT_DIR),
    "seeds": SEEDS,
    "device": str(DEVICE),
    "n_main": N_MAIN,
    "d_main": D_MAIN,
    "n_rules": N_RULES,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "l1_mask": L1_MASK,
    "rule_beta": RULE_BETA,
    "tables": sorted([p.name for p in TABLE_DIR.glob("*.csv")]),
    "figures": sorted([p.name for p in FIG_DIR.glob("*.png")]),
}

with open(OUTPUT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\nFiles saved:")
for fname in manifest["tables"]:
    print(f"  tables/{fname}")

for fname in manifest["figures"]:
    print(f"  figures/{fname}")

print("\nManifest:")
print(json.dumps(manifest, indent=2))


Files saved:
  tables/paper_table_main_synthetic_benchmark.csv
  tables/table_environment.csv
  tables/table_failed_runs.csv
  tables/table_hyperparameters_synthetic.csv
  tables/table_model_summary.csv
  tables/table_per_dataset_summary.csv
  tables/table_per_dataset_winners.csv
  tables/table_ranked_results.csv
  tables/table_s1_dataset_specification.csv
  tables/table_s2_benchmark_raw_results.csv
  tables/table_s5_extracted_smte_gmte_rules.csv
  tables/table_top_features.csv
  figures/figure_1_accuracy_interpretability_frontier.png
  figures/figure_2_performance_heatmap.png
  figures/figure_3_interpretability_decomposition.png
  figures/figure_4_feature_recovery_heatmap.png
  figures/figure_5_model_card_radar.png
  figures/figure_6_rule_structure_Hierarchical.png
  figures/figure_7_training_curves.png

Manifest:
{
  "output_dir": "outputs_synthetic_paper",
  "seeds": [
    1,
    12,
    123,
    1234,
    12345
  ],
  "device": "cuda",
  "n_main": 2000,
  "d_main": 15,
  "n_rules"